# 01 — Transformer Primer (for mech interp)

You know regression, neural nets, and a bit of PPO — this notebook fills the
specific gap: how a decoder-only transformer (GPT-2) is put together, using
the vocabulary `TransformerLens` uses internally. Every later notebook leans
on the names introduced here.

**The core idea of mech interp**: a transformer's forward pass is not a black
box — it's a sequence of read/write operations on a shared vector called the
**residual stream**. Every component (attention head, MLP) reads from it and
writes back to it. Interpretability = figuring out what each read/write is
doing.


## Glossary — every abbreviation used below

Mech interp code leans hard on short variable names because you're constantly
writing tensor shapes in comments. Here's every one you'll hit in this
notebook, spelled out, so nothing is just pattern-matched:

| Shorthand | Full name | What it actually means |
|---|---|---|
| `d_model` | model dimension | size of each vector in the residual stream. 768 for GPT-2 small — every token position holds a 768-number vector at all times. |
| `d_head` | head dimension | size of the vector each individual attention head works with internally (64 here). `n_heads * d_head` = `d_model` (12 × 64 = 768) — the residual stream is literally split evenly across heads. |
| `d_mlp` | MLP hidden dimension | size of the hidden layer inside the MLP block (3072 = 4 × `d_model`, GPT-2's standard ratio). |
| `d_vocab` | vocabulary size | number of distinct tokens the tokenizer knows (50257 for GPT-2). Logits have this many entries — one score per possible next token. |
| `n_layers` | number of layers | how many stacked transformer blocks (attention + MLP pairs). 12 for GPT-2 small. |
| `n_heads` | number of attention heads | how many independent attention "sub-processes" run in parallel inside each layer's attention block. 12 here. |
| `n_ctx` | context length | max number of token positions the model can process at once (1024). |
| `MLP` | multi-layer perceptron | the plain feedforward block in each layer (linear → nonlinearity → linear). "MLP" here just names *that specific sub-block*, not the general ML concept. |
| `LN` / `ln1` / `ln2` | LayerNorm | a normalization step applied before attention (`ln1`) and before the MLP (`ln2`) in each layer. Keeps activation scales stable; doesn't add information itself. |
| `Q`, `K`, `V` | Query, Key, Value | the three vectors each attention head computes per token. Loosely: Q = "what am I looking for," K = "what do I contain," V = "what do I broadcast if attended to." `hook_q`/`hook_k`/`hook_v` are the cached versions of each. |
| `attn` | attention | shorthand for the attention block/mechanism throughout hook names (`blocks.0.attn.hook_pattern`, etc.). |
| `resid` | residual (stream) | shorthand for the residual stream itself, as in `resid_pre`, `resid_mid`, `resid_post`. |
| `pos` / `position` | token position | index of a token in the sequence (0, 1, 2, ...). |
| `dest` / `src` | destination / source | in an attention pattern, `dest` is the position doing the attending (asking "what should I look at?"), `src` is the position being attended to (being looked at). |
| `batch` | batch | how many independent sequences are processed in parallel in one tensor (an unrelated-to-content dimension, like in any PyTorch code). |
| `cfg` | config | the `HookedTransformerConfig` object holding all the above as named fields (`model.cfg.d_model`, etc.). |
| `BOS` | beginning-of-sequence token | a special token (`<|endoftext|>` for GPT-2) TransformerLens prepends by default so the first real token has something to attend back to. |
| `logits` | — (not an abbreviation, but jargon) | the raw, pre-softmax output scores over the vocabulary; higher = model considers that token a more likely next token. |
| `GELU` (`act_fn: gelu_new`) | Gaussian Error Linear Unit | the nonlinearity used inside the MLP block (a smoother alternative to ReLU). |

You don't need to memorize this table — refer back to it as these names show
up below and in later notebooks.


In [1]:
import torch
from transformer_lens import HookedTransformer, utils
import plotly.express as px

torch.set_grad_enabled(False)  # we're only doing inference/analysis here
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


/tmp/ipykernel_67909/3054563688.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


In [2]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
print(model.cfg)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 15306.55it/s]


Loaded pretrained model gpt2 into HookedTransformer
HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_

That config dump has ~60 fields because `HookedTransformerConfig` is shared
across many model architectures (LLaMA, Mistral, mixture-of-experts, etc.),
so most fields are irrelevant to plain GPT-2 and stay at their default
(`None`, `False`, `-1.0`...). The ones that matter for us, all defined in the
glossary above: `d_model=768`, `n_layers=12`, `n_heads=12`, `d_head=64`,
`d_mlp=3072`, `d_vocab=50257`, `n_ctx=1024`, `act_fn='gelu_new'`. Everything
about RoPE/NTK/YaRN (`rotary_*`, `NTK_*`, `yarn_*`) is for *other* models'
positional encodings — GPT-2 uses `positional_embedding_type: 'standard'`
(plain learned position embeddings) instead, so those fields are unused here.


## The residual stream

Shape: `[batch, position, d_model]`. Think of it as a shared scratchpad with
`d_model` (768 for GPT-2 small) "slots" at every token position. Each layer
does:

```
resid_pre -> [Attention] -> resid_mid -> [MLP] -> resid_post
```

and `resid_post` of layer *n* becomes `resid_pre` of layer *n+1*. Nothing is
ever overwritten — components only *add* to the stream. That additive
structure is what makes the whole field tractable: you can decompose the
final output as a sum of every component's individual contribution.


In [3]:
text = "The quick brown fox jumps over the lazy dog"
tokens = model.to_tokens(text)
print(tokens.shape, tokens)
print([model.to_string(t) for t in tokens[0]])


torch.Size([1, 10]) tensor([[50256,   464,  2068,  7586, 21831, 18045,   625,   262, 16931,  3290]],
       device='cuda:0')
['<|endoftext|>', 'The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog']


## `run_with_cache`

This is the single most important TransformerLens method. It runs a forward
pass and saves every intermediate activation, keyed by name. Names follow the
pattern `blocks.{layer}.{component}.{hook_point}` (a "hook" is just a named
tap point in the computation graph — TransformerLens inserts one after every
step worth inspecting). Some you'll see in the next cell's output:

- `hook_embed` — the token embedding: each token id looked up in the embedding matrix (named `W_E`, covered properly in notebook 01) to get its starting residual-stream vector
- `hook_pos_embed` — the positional embedding added on top of it (a separate learned vector per position, so the model knows *where* each token is)
- `blocks.0.hook_resid_pre` — residual stream entering layer 0 (`resid` = residual stream, `pre` = before this layer's attention runs)
- `blocks.0.ln1.hook_scale` / `hook_normalized` — the LayerNorm (`ln1`, applied before attention) recorded as "how much did it scale by" and "the result after normalizing"
- `blocks.0.attn.hook_q` / `hook_k` / `hook_v` — the Query/Key/Value vectors each head computed, shape `[batch, pos, n_heads, d_head]`
- `blocks.0.attn.hook_attn_scores` — raw Q·K similarity scores before the softmax, shape `[batch, head, dest_pos, src_pos]`
- `blocks.0.attn.hook_pattern` — those scores *after* softmax, i.e. the actual attention weights (sum to 1 across `src_pos` for each `dest_pos`), same shape
- `blocks.0.attn.hook_z` — each head's output: the Values (`V`) mixed together according to `hook_pattern`, still kept *per-head* (shape `[batch, pos, n_heads, d_head]`) rather than merged back into one `d_model`-sized vector — that merge happens in a separate step right after
- `blocks.0.hook_mlp_out` — what the MLP (feedforward block) added to the stream at this layer


In [4]:
logits, cache = model.run_with_cache(tokens)
print("logits shape:", logits.shape)  # [batch, position, d_vocab]
print(list(cache.keys())[:10])


logits shape: torch.Size([1, 10, 50257])
['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern']


## Attention vs. MLP, in one sentence each

- **Attention** moves information *between positions*: each head decides,
  per destination position, which source positions to read from
  (`hook_pattern`) and what to copy (`hook_z`). It's the only component that
  lets one token "see" another.
- **MLP** acts *independently per position*: it reads the residual stream at
  a single position and writes back a nonlinear function of it. No
  cross-token communication happens here.

Everything you'll do in later notebooks — induction heads, logit lens,
activation patching — is just closely reading these two operations across
layers.


In [5]:
# quick look: what does the attention pattern of layer 0 look like?
pattern = cache["blocks.0.attn.hook_pattern"]  # [batch, head, dest, src]
print(pattern.shape)
str_tokens = model.to_str_tokens(text)
px.imshow(pattern[0, 0].cpu(), x=str_tokens, y=str_tokens,
          labels=dict(x="source", y="destination"),
          title="Layer 0, Head 0 attention pattern").show()


torch.Size([1, 12, 10, 10])


## Next

`02_transformerlens_basics.ipynb` — same tools, more exploration: unembedding,
`utils.get_act_name`, and `circuitsvis` for nicer attention visualizations.
